In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch07_2. 다중 클래스 분류 (와인품질등급예측)</font>

# 레드 와인 품질 등급 예측
```
1. 데이터셋 확보 & 전처리
    결측치 처리 >> 독립변수와 종속변수 분리 >> 독립변수의 스케일 조정(std) >> 종속변수의 균형 확인
    >> 종속변수 원-핫 인코딩(get_dummies) >> 훈련 데이터셋과 검증 데이터셋 분리(train_test_split)
2. 모델 구성
3. 모델 학습과정 설정 (다중 분류)
4. 모델 학습 (callbacks, compute_class_weight 활용)
5. 모델 평가 (시각화, 평가, 교차표)
6. 모델 사용 (저장, 예측)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.layers import ReLU, ELU, LeakyReLU
from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import HeNormal # 카이밍 초기화

from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.metrics import Recall, Precision
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix

## 1. 데이터셋 확보 및 전처리
- 독립변수와 종속변수 분리
- 독립변수의 스케일 조정(std)
- 종속변수의 균형 확인
- 종속변수 원-핫 인코딩(get_dummies)
- 훈련 데이터셋과 검증 데이터셋 분리(train_test_split)

In [ ]:
redwine = pd.read_csv('data/winequality-red.csv', delimiter=';')
redwine.info()

In [ ]:
# 독립변수와 종속변수 분리
X_redwine = redwine.iloc[:,:-1].to_numpy()
y_redwine = redwine.iloc[:,-1]
X_redwine.shape, y_redwine.shape

In [ ]:
# 독립변수의 스케일 조정
scaler = StandardScaler()

redwine_X = scaler.fit_transform(X_redwine)

In [ ]:
# 종속변수의 원-핫 인코딩
redwine_Y = pd.get_dummies(y_redwine)

In [ ]:
# 종속변수의 균형 확인
y_redwine.value_counts(normalize=True)

In [ ]:
# 독립변수와 타겟변수
redwine_X.shape, redwine_Y.shape

In [ ]:
# 훈련 데이터셋과 검증 데이터셋 분리
X_train, X_test, Y_train, Y_test = train_test_split(redwine_X, redwine_Y, test_size=0.3, stratify=redwine_Y, random_state=12)

In [ ]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

## 2. 모델 구성
- 입력 : 11 >> 64 >> 50 >> 30 >> 출력 : 6
- ReLU 활성화 함수와 He Normal 가중치 초기화 : '가중치 소멸'과 '신호의 폭주/줄어듦'을 방지

In [ ]:
model = Sequential(name='redwine_seq')
model.add(Input(shape=(11,)))

model.add(Dense(units=128, activation='elu', kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(units=64, activation='relu', kernel_initializer=HeNormal(seed=42)))
model.add(Dropout(0.3))

model.add(Dense(units=32, activation='relu', kernel_initializer='he_normal')) # relu를 사용할 때, he_normal을 함께 사용
model.add(Dropout(0.3))

model.add(Dense(units=16, activation='elu', kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(units=6, activation='softmax'))
model.summary()

## 3. 모델 학습과정 설정

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy', Recall(name='recall'), Precision(name='precision')])

## 4. 모델 학습
- callbacks 활용

In [ ]:
%%time
earlystop = EarlyStopping(monitor='val_loss', patience=50, verbose=0)

hist = model.fit(X_train, Y_train,
                epochs=200,
                batch_size=128,
                validation_split=0.2,
                callbacks=[earlystop],
                verbose=1)

## 5. 모델 평가

In [ ]:
fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_accuracy'], 'b', label='val_accuracy')
acc_ax.plot(hist.history['val_precision'], 'k', label='val_precision')
acc_ax.plot(hist.history['val_recall'], 'm', label='val_recall')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.6), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.8), ncol=1)
plt.show()

In [ ]:
loss, accuracy, recall, precision = model.evaluate(X_test, Y_test, verbose=0)

In [ ]:
f1_score = 2 * (precision * recall) / (precision + recall)

print(f"정확도 : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"정밀도 : {precision:.4f} ({precision*100:.2f}%)")
print(f"재현율 : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score : {f1_score:.4f} ({f1_score*100:.2f}%)")

In [ ]:
y_hat = model.predict(X_test).argmax(axis=1) + 3
y_test = Y_test.values.argmax(axis=1) + 3

In [ ]:
pd.crosstab(y_test, y_hat, rownames=['실제값'], colnames=['예측값'])

## 6. 모델 사용

In [ ]:
X_redwine[0]

In [ ]:
data = [[ 7.4,  0.7,  0,  1.9,  0.076, 11, 34,0.9978,  3.51,  0.56,  9.4]]
input_data = scaler.transform(data)
model.predict(input_data).argmax(axis=1) + 3